**Cell 1 — Load the saved artifacts**

In [1]:
import numpy as np
import tensorflow as tf
import json
from pathlib import Path

# Project paths
PROJECT_ROOT = Path(r"D:\PROJECTS\64feature_NN\RGB_NN")
FEATURE_DIR = PROJECT_ROOT / "features"

# --------------------------------------------------
# Load trained model
# --------------------------------------------------

MODEL_PATH = FEATURE_DIR / "fod_mlp.keras"

model = tf.keras.models.load_model(MODEL_PATH)

print("Model loaded successfully")
model.summary()

Model loaded successfully


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 4)                   │              68 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 8,030 (31.37 KB)

 Trainable params: 2,676 (10.45 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,354 (20.92 KB)

In [2]:
# --------------------------------------------------
# Load saved datasets
# --------------------------------------------------

X_train_int8 = np.load(FEATURE_DIR / "X_train_int8.npy")
X_val_int8   = np.load(FEATURE_DIR / "X_val_int8.npy")
X_test_int8  = np.load(FEATURE_DIR / "X_test_int8.npy")

y_train = np.load(FEATURE_DIR / "y_train.npy")
y_val   = np.load(FEATURE_DIR / "y_val.npy")
y_test  = np.load(FEATURE_DIR / "y_test.npy")

print("X_train_int8:", X_train_int8.shape, X_train_int8.dtype)
print("X_val_int8:  ", X_val_int8.shape, X_val_int8.dtype)
print("X_test_int8: ", X_test_int8.shape, X_test_int8.dtype)

print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

X_train_int8: (1870, 64) int8
X_val_int8:   (384, 64) int8
X_test_int8:  (412, 64) int8
y_train: (1870,)
y_val:   (384,)
y_test:  (412,)


**Inspect the trained MLP**

In [3]:
# --------------------------------------------------
# Extract trained MLP weights and biases
# --------------------------------------------------

print("\nTRAINED MODEL LAYERS")
print("=" * 50)

for i, layer in enumerate(model.layers):
    print(f"\nLayer {i}: {layer.name}")
    print(f"Type: {type(layer).__name__}")

    weights = layer.get_weights()

    if len(weights) == 0:
        print("No trainable weights")
        continue

    W, b = weights

    print("Weights shape :", W.shape)
    print("Bias shape    :", b.shape)

    print("Weights dtype :", W.dtype)
    print("Bias dtype    :", b.dtype)

    print("Weight range  :", W.min(), "to", W.max())
    print("Bias range    :", b.min(), "to", b.max())


TRAINED MODEL LAYERS

Layer 0: dense
Type: Dense
Weights shape : (64, 32)
Bias shape    : (32,)
Weights dtype : float32
Bias dtype    : float32
Weight range  : -0.31652164 to 0.31981605
Bias range    : -0.0383602 to 0.070709735

Layer 1: dense_1
Type: Dense
Weights shape : (32, 16)
Bias shape    : (16,)
Weights dtype : float32
Bias dtype    : float32
Weight range  : -0.39914963 to 0.426456
Bias range    : -0.035493575 to 0.090999536

Layer 2: dense_2
Type: Dense
Weights shape : (16, 4)
Bias shape    : (4,)
Weights dtype : float32
Bias dtype    : float32
Weight range  : -0.5677501 to 0.50287575
Bias range    : -0.025651688 to 0.02547786


**Recover the exact input quantization scale**

In [4]:
# --------------------------------------------------
# Inspect feature scaler
# --------------------------------------------------

scaler_data = np.load(FEATURE_DIR / "scaler.npz")

print("Scaler contents:")
print(scaler_data.files)

for key in scaler_data.files:
    arr = scaler_data[key]
    print(f"\n{key}")
    print("Shape:", arr.shape)
    print("Dtype:", arr.dtype)
    print("Min:", arr.min())
    print("Max:", arr.max())

Scaler contents:
['mean', 'std', 'clip']

mean
Shape: (64,)
Dtype: float32
Min: -12.445952
Max: 60.735115

std
Shape: (64,)
Dtype: float32
Min: 0.010335202
Max: 16.079832

clip
Shape: ()
Dtype: float32
Min: 4.0
Max: 4.0


In [5]:
# --------------------------------------------------
# Inspect INT8 quantization
# --------------------------------------------------

print("\nINT8 DATA RANGE")
print("=" * 40)

for name, X in [
    ("Train", X_train_int8),
    ("Validation", X_val_int8),
    ("Test", X_test_int8)
]:
    print(
        f"{name:10s}: "
        f"min={X.min():4d}, "
        f"max={X.max():4d}, "
        f"mean={X.mean():.3f}, "
        f"dtype={X.dtype}"
    )


INT8 DATA RANGE
Train     : min=-127, max= 127, mean=-0.038, dtype=int8
Validation: min=-127, max= 127, mean=-0.259, dtype=int8
Test      : min=-127, max= 127, mean=-1.204, dtype=int8


**Verify the quantization mathematically**

In [6]:
# --------------------------------------------------
# Verify INT8 quantization scale
# --------------------------------------------------

INPUT_SCALE = 4.0 / 127.0

print("Input scale:", INPUT_SCALE)

# Load the original standardized features
X_train_std = np.load(FEATURE_DIR / "X_train_std.npy")

# Recreate INT8 quantization
X_train_reconstructed = np.clip(
    X_train_std,
    -4.0,
    4.0
)

X_train_reconstructed = np.round(
    X_train_reconstructed * (127.0 / 4.0)
).astype(np.int8)

# Compare against saved INT8 data
difference = (
    X_train_reconstructed.astype(np.int16)
    - X_train_int8.astype(np.int16)
)

print("\nVerification")
print("=" * 40)
print("Maximum difference:", np.max(np.abs(difference)))
print("Mean absolute difference:", np.mean(np.abs(difference)))
print("Number of different values:", np.count_nonzero(difference))
print("Total values:", difference.size)

Input scale: 0.031496062992125984

Verification
Maximum difference: 0
Mean absolute difference: 0.0
Number of different values: 0
Total values: 119680


**Quantize the MLP weights**

In [8]:
# --------------------------------------------------
# Quantize MLP weights to INT8
# --------------------------------------------------

def quantize_weights_int8(W):
    """
    Symmetric per-layer INT8 quantization.
    """

    max_abs = np.max(np.abs(W))

    scale = max_abs / 127.0

    W_int8 = np.round(W / scale)
    W_int8 = np.clip(W_int8, -127, 127).astype(np.int8)

    return W_int8, scale


# Extract trained weights
W1, b1 = model.layers[0].get_weights()
W2, b2 = model.layers[1].get_weights()
W3, b3 = model.layers[2].get_weights()


# Quantize
W1_int8, S_W1 = quantize_weights_int8(W1)
W2_int8, S_W2 = quantize_weights_int8(W2)
W3_int8, S_W3 = quantize_weights_int8(W3)


# --------------------------------------------------
# Display results
# --------------------------------------------------

print("WEIGHT QUANTIZATION")
print("=" * 60)

print("\nLayer 1")
print("FP32 shape :", W1.shape)
print("FP32 range :", W1.min(), "to", W1.max())
print("Scale      :", S_W1)
print("INT8 range :", W1_int8.min(), "to", W1_int8.max())

print("\nLayer 2")
print("FP32 shape :", W2.shape)
print("FP32 range :", W2.min(), "to", W2.max())
print("Scale      :", S_W2)
print("INT8 range :", W2_int8.min(), "to", W2_int8.max())

print("\nLayer 3")
print("FP32 shape :", W3.shape)
print("FP32 range :", W3.min(), "to", W3.max())
print("Scale      :", S_W3)
print("INT8 range :", W3_int8.min(), "to", W3_int8.max())

WEIGHT QUANTIZATION

Layer 1
FP32 shape : (64, 32)
FP32 range : -0.31652164 to 0.31981605
Scale      : 0.0025182366
INT8 range : -126 to 127

Layer 2
FP32 shape : (32, 16)
FP32 range : -0.39914963 to 0.426456
Scale      : 0.0033579213
INT8 range : -119 to 127

Layer 3
FP32 shape : (16, 4)
FP32 range : -0.5677501 to 0.50287575
Scale      : 0.004470473
INT8 range : -127 to 112


**measure the weight quantization error**

In [9]:
# --------------------------------------------------
# Measure weight quantization error
# --------------------------------------------------

def dequantize_weights(W_int8, scale):
    return W_int8.astype(np.float32) * scale


W1_deq = dequantize_weights(W1_int8, S_W1)
W2_deq = dequantize_weights(W2_int8, S_W2)
W3_deq = dequantize_weights(W3_int8, S_W3)


def print_quant_error(name, W, W_deq):
    error = W_deq - W

    print(f"\n{name}")
    print("-" * 40)
    print("MAE     :", np.mean(np.abs(error)))
    print("RMSE    :", np.sqrt(np.mean(error ** 2)))
    print("Max err :", np.max(np.abs(error)))


print_quant_error("Layer 1", W1, W1_deq)
print_quant_error("Layer 2", W2, W2_deq)
print_quant_error("Layer 3", W3, W3_deq)


Layer 1
----------------------------------------
MAE     : 0.0006314934
RMSE    : 0.00073205074
Max err : 0.0012590587

Layer 2
----------------------------------------
MAE     : 0.0008276614
RMSE    : 0.00095374655
Max err : 0.0016744286

Layer 3
----------------------------------------
MAE     : 0.0010468676
RMSE    : 0.0012417691
Max err : 0.0022189915


**Cell 7: First integer MAC reference**

**Let's start with only the first layer.**

In [10]:
# --------------------------------------------------
# Integer MAC reference - Layer 1
# --------------------------------------------------

def integer_dense_layer(X_int8, W_int8):
    """
    Integer-only matrix multiplication.

    X_int8 : (N, input_features)
    W_int8 : (input_features, output_features)

    Returns:
        INT32 accumulator
    """

    X_int32 = X_int8.astype(np.int32)
    W_int32 = W_int8.astype(np.int32)

    return X_int32 @ W_int32


# Test on the complete test set
Z1_int = integer_dense_layer(
    X_test_int8,
    W1_int8
)

print("Integer Layer 1")
print("=" * 50)

print("Shape :", Z1_int.shape)
print("Dtype :", Z1_int.dtype)
print("Min   :", Z1_int.min())
print("Max   :", Z1_int.max())
print("Mean  :", Z1_int.mean())

Integer Layer 1
Shape : (412, 32)
Dtype : int32
Min   : -78400
Max   : 68827
Mean  : 234.78231189320388


**Compare integer Layer 1 with FP32 Layer 1**

In [11]:
# --------------------------------------------------
# Compare integer Layer 1 with FP32 Layer 1
# --------------------------------------------------

# Input scale
S_X = 4.0 / 127.0

# Layer 1 output scale
S_Z1 = S_X * S_W1

print("Layer 1 output scale:", S_Z1)

# Convert integer accumulator back to FP32
Z1_int_dequant = Z1_int.astype(np.float32) * S_Z1

# Original TensorFlow Layer 1 computation
Z1_fp32 = X_test_int8.astype(np.float32) * S_X
Z1_fp32 = Z1_fp32 @ W1

# Compare
error = Z1_int_dequant - Z1_fp32

print("\nLayer 1 comparison")
print("=" * 50)

print("FP32 output range:")
print("Min:", Z1_fp32.min())
print("Max:", Z1_fp32.max())

print("\nDequantized INT8 output range:")
print("Min:", Z1_int_dequant.min())
print("Max:", Z1_int_dequant.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(error)))
print("RMSE    :", np.sqrt(np.mean(error ** 2)))
print("Max err :", np.max(np.abs(error)))

Layer 1 output scale: 7.931454e-05

Layer 1 comparison
FP32 output range:
Min: -6.2280827
Max: 5.452918

Dequantized INT8 output range:
Min: -6.21826
Max: 5.4589815

Quantization error:
MAE     : 0.0041577383
RMSE    : 0.005259001
Max err : 0.026161075


Excellent. This is another successful checkpoint. ✅

Your first integer layer is behaving very well.

**Quantize the biases**

In [12]:
# --------------------------------------------------
# Quantize biases into accumulator scale
# --------------------------------------------------

S_X = 4.0 / 127.0

# Accumulator/output scales for each layer
S_Z1 = S_X * S_W1

# Bias scale for Layer 1
S_B1 = S_Z1

b1_int32 = np.round(
    b1 / S_B1
).astype(np.int32)

print("Layer 1 bias quantization")
print("=" * 50)

print("Bias scale:", S_B1)

print("\nFP32 bias:")
print("Min:", b1.min())
print("Max:", b1.max())

print("\nINT32 bias:")
print("Min:", b1_int32.min())
print("Max:", b1_int32.max())
print("Dtype:", b1_int32.dtype)

# Dequantize and measure error
b1_deq = b1_int32.astype(np.float32) * S_B1

bias_error = b1_deq - b1

print("\nBias quantization error:")
print("MAE     :", np.mean(np.abs(bias_error)))
print("RMSE    :", np.sqrt(np.mean(bias_error ** 2)))
print("Max err :", np.max(np.abs(bias_error)))

Layer 1 bias quantization
Bias scale: 7.931454e-05

FP32 bias:
Min: -0.0383602
Max: 0.070709735

INT32 bias:
Min: -484
Max: 892
Dtype: int32

Bias quantization error:
MAE     : 2.1972097e-05
RMSE    : 2.5039748e-05
Max err : 3.9234757e-05


**Complete Layer 1 integer implementation**

In [13]:
# --------------------------------------------------
# Complete integer Layer 1
# MAC + bias + ReLU
# --------------------------------------------------

# Integer MAC
Z1_int = X_test_int8.astype(np.int32) @ W1_int8.astype(np.int32)

# Add quantized bias
Z1_int_bias = Z1_int + b1_int32

# ReLU
A1_int = np.maximum(Z1_int_bias, 0)

print("Integer Layer 1")
print("=" * 50)

print("Shape :", A1_int.shape)
print("Dtype :", A1_int.dtype)
print("Min   :", A1_int.min())
print("Max   :", A1_int.max())
print("Mean  :", A1_int.mean())

Integer Layer 1
Shape : (412, 32)
Dtype : int32
Min   : 0
Max   : 69390
Mean  : 8465.907994538835


In [14]:
# --------------------------------------------------
# Compare complete Layer 1 with TensorFlow
# --------------------------------------------------

A1_int_dequant = A1_int.astype(np.float32) * S_Z1

# Original FP32 TensorFlow Layer 1
X_test_std = np.load(FEATURE_DIR / "X_test_std.npy")

Z1_fp32_full = X_test_std @ W1 + b1
A1_fp32 = np.maximum(Z1_fp32_full, 0)

error = A1_int_dequant - A1_fp32

print("\nLayer 1 complete comparison")
print("=" * 50)

print("FP32 activation range:")
print("Min:", A1_fp32.min())
print("Max:", A1_fp32.max())

print("\nINT8/INT32 dequantized activation range:")
print("Min:", A1_int_dequant.min())
print("Max:", A1_int_dequant.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(error)))
print("RMSE    :", np.sqrt(np.mean(error ** 2)))
print("Max err :", np.max(np.abs(error)))


Layer 1 complete comparison
FP32 activation range:
Min: 0.0
Max: 5.498295

INT8/INT32 dequantized activation range:
Min: 0.0
Max: 5.503636

Quantization error:
MAE     : 0.0048356075
RMSE    : 0.008542949
Max err : 0.049775362


✅ Layer 1 is validated end-to-end.
Input quantization       ✅ VERIFIED
Weight quantization      ✅ VERIFIED
Layer 1 integer MAC      ✅ VERIFIED
Layer 1 bias             ✅ VERIFIED
Layer 1 ReLU             ✅ VERIFIED

**Analyze Layer 1 activation**
This will tell us whether the Layer 1 activation is mostly concentrated near zero or whether we need a wider INT8 range.

In [16]:
# --------------------------------------------------
# Analyze Layer 1 activation for INT8 requantization
# --------------------------------------------------

print("Layer 1 FP32 activation statistics")
print("=" * 50)

print("Min :", A1_fp32.min())
print("Max :", A1_fp32.max())
print("Mean:", A1_fp32.mean())
print("Std :", A1_fp32.std())

# Percentiles
percentiles = [50, 90, 95, 99, 99.9, 100]

print("\nPercentiles:")
for p in percentiles:
    print(f"{p:5.1f}% :", np.percentile(A1_fp32, p))

Layer 1 FP32 activation statistics
Min : 0.0
Max : 5.498295
Mean: 0.67177314
Std : 0.9973405

Percentiles:
 50.0% : 0.010660358
 90.0% : 2.0510056
 95.0% : 2.7853305
 99.0% : 4.1464276
 99.9% : 5.0842276
100.0% : 5.498295


**First, let's test the straightforward full-range scale**

In [17]:
# --------------------------------------------------
# Layer 1 activation → INT8
# Full-range activation quantization
# --------------------------------------------------

A1_max = np.max(A1_fp32)

S_A1 = A1_max / 127.0

A1_int8 = np.round(
    A1_fp32 / S_A1
)

A1_int8 = np.clip(
    A1_int8,
    0,
    127
).astype(np.int8)

# Dequantize
A1_dequant = A1_int8.astype(np.float32) * S_A1

# Error
A1_error = A1_dequant - A1_fp32

print("Layer 1 activation quantization")
print("=" * 50)

print("Activation max :", A1_max)
print("Activation scale:", S_A1)

print("\nINT8 range:")
print("Min:", A1_int8.min())
print("Max:", A1_int8.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(A1_error)))
print("RMSE    :", np.sqrt(np.mean(A1_error ** 2)))
print("Max err :", np.max(np.abs(A1_error)))

print("\nSaturation count:")
print("127 values:", np.sum(A1_int8 == 127))

Layer 1 activation quantization
Activation max : 5.498295
Activation scale: 0.04329366

INT8 range:
Min: 0
Max: 127

Quantization error:
MAE     : 0.00545881
RMSE    : 0.008880993
Max err : 0.021642745

Saturation count:
127 values: 2


Perfect. ✅ The Layer 1 activation quantization is also behaving well.

Result
$$ S_{A1}=\frac{5.498295}{127}=\boxed{0.04329366} $$

Your conversion gives:

Metric	Result
FP32 activation range	0 → 5.4983
INT8 range	0 → 127
Activation scale	0.04329366
MAE	0.00546
RMSE	0.00888
Max error	0.02164
Saturated values	2

Only 2 out of 13,184 activations hit 127, so we're barely losing anything to saturation.

**Build Layer 2 completely in integer arithmetic**

In [18]:
# --------------------------------------------------
# Layer 2 integer scales and bias
# --------------------------------------------------

S_Z2 = S_A1 * S_W2

b2_int32 = np.round(
    b2 / S_Z2
).astype(np.int32)

print("Layer 2")
print("=" * 50)

print("Activation scale S_A1 :", S_A1)
print("Weight scale S_W2    :", S_W2)
print("Accumulator scale    :", S_Z2)

print("\nFP32 bias:")
print("Min:", b2.min())
print("Max:", b2.max())

print("\nINT32 bias:")
print("Min:", b2_int32.min())
print("Max:", b2_int32.max())

# Bias reconstruction error
b2_deq = b2_int32.astype(np.float32) * S_Z2
b2_error = b2_deq - b2

print("\nBias quantization error:")
print("MAE     :", np.mean(np.abs(b2_error)))
print("RMSE    :", np.sqrt(np.mean(b2_error ** 2)))
print("Max err :", np.max(np.abs(b2_error)))

Layer 2
Activation scale S_A1 : 0.04329366
Weight scale S_W2    : 0.0033579213
Accumulator scale    : 0.0001453767

FP32 bias:
Min: -0.035493575
Max: 0.090999536

INT32 bias:
Min: -244
Max: 626

Bias quantization error:
MAE     : 3.3758522e-05
RMSE    : 4.0962415e-05
Max err : 7.239729e-05


In [19]:
# --------------------------------------------------
# Integer Layer 2
# INT8 activation × INT8 weights
# + INT32 bias
# --------------------------------------------------

Z2_int = (
    A1_int8.astype(np.int32)
    @ W2_int8.astype(np.int32)
)

Z2_int_bias = Z2_int + b2_int32

A2_int = np.maximum(
    Z2_int_bias,
    0
)

print("\nInteger Layer 2")
print("=" * 50)

print("Shape :", A2_int.shape)
print("Dtype :", A2_int.dtype)
print("Min   :", A2_int.min())
print("Max   :", A2_int.max())
print("Mean  :", A2_int.mean())


Integer Layer 2
Shape : (412, 16)
Dtype : int32
Min   : 0
Max   : 37590
Mean  : 6224.627275485437


In [20]:
# --------------------------------------------------
# Compare Layer 2 against TensorFlow
# --------------------------------------------------

# Dequantize integer Layer 2
A2_int_dequant = (
    A2_int.astype(np.float32) * S_Z2
)

# Original FP32 Layer 2
Z2_fp32 = A1_fp32 @ W2 + b2
A2_fp32 = np.maximum(
    Z2_fp32,
    0
)

error = A2_int_dequant - A2_fp32

print("\nLayer 2 comparison")
print("=" * 50)

print("FP32 activation range:")
print("Min:", A2_fp32.min())
print("Max:", A2_fp32.max())

print("\nINT8/INT32 dequantized range:")
print("Min:", A2_int_dequant.min())
print("Max:", A2_int_dequant.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(error)))
print("RMSE    :", np.sqrt(np.mean(error ** 2)))
print("Max err :", np.max(np.abs(error)))


Layer 2 comparison
FP32 activation range:
Min: 0.0
Max: 5.45245

INT8/INT32 dequantized range:
Min: 0.0
Max: 5.46471

Quantization error:
MAE     : 0.0046593207
RMSE    : 0.008309941
Max err : 0.04261136


**Layer 2 is also validated.** ✅

We now have two complete integer layers that closely reproduce the original FP32 network.

Layer 2 results
Quantity	Result
Activation scale \(S_{A1}\)	0.04329366
Weight scale \(S_{W2}\)	0.0033579213
Accumulator scale \(S_{Z2}\)	0.0001453767
INT32 bias range	−244 to +626
Bias MAE	\(3.38\times10^{-5}\)
Activation MAE	0.00466
Activation RMSE	0.00831
Maximum error	0.04261

The integer Layer 2 output is:

Shape : (412, 16)
Dtype : int32
Range : 0 → 37590

and after dequantization:

FP32 range       : 0 → 5.45245
Integer equivalent: 0 → 5.46471

So the accumulated quantization error is still small.

Current integer pipeline

We have now established:

                 INPUT
                   │
             INT8 features
             Sx = 0.0314961
                   │
                   ▼
          ┌─────────────────┐
          │ Dense 64 → 32   │
          │ W1 = INT8       │
          │ B1 = INT32      │
          │ accumulator     │
          └────────┬────────┘
                   │
                  ReLU
                   │
             INT8 activation
             SA1 = 0.0432937
                   │
                   ▼
          ┌─────────────────┐
          │ Dense 32 → 16   │
          │ W2 = INT8       │
          │ B2 = INT32      │
          │ accumulator     │
          └────────┬────────┘
                   │
                  ReLU
                   │
                INT32

The only remaining neural-network layer is:

$$ 16\rightarrow4 $$

Then we'll have the complete integer network.

# Final Layer

The final layer is slightly different because it has no ReLU:

Dense 16 → 4
       ↓
    logits
       ↓
   Softmax
       ↓
   class ID

For FPGA inference, we actually do not need to calculate the floating-point softmax just to determine the class.

Because softmax preserves the ordering of logits:

$$ \operatorname{argmax}(\text{softmax}(z)) = \operatorname{argmax}(z) $$

So ultimately we'll be able to do:

INT32 logits
     ↓
argmax
     ↓
class ID

That is very useful for the FPGA implementation.

First, quantize Layer 3 bias

Run:

In [23]:
# --------------------------------------------------
# Layer 3 integer scales and bias
# --------------------------------------------------

S_Z3 = S_A2 * S_W3 if "S_A2" in globals() else None

print("Current variables:")
print("S_Z3:", S_Z3)

Current variables:
S_Z3: None


**analyze Layer 2's FP32 activation before deciding how to represent it as INT8.**

In [24]:
# --------------------------------------------------
# Analyze Layer 2 activation
# --------------------------------------------------

print("Layer 2 FP32 activation statistics")
print("=" * 50)

print("Min :", A2_fp32.min())
print("Max :", A2_fp32.max())
print("Mean:", A2_fp32.mean())
print("Std :", A2_fp32.std())

percentiles = [50, 90, 95, 99, 99.9, 100]

print("\nPercentiles:")

for p in percentiles:
    print(f"{p:5.1f}% :", np.percentile(A2_fp32, p))

Layer 2 FP32 activation statistics
Min : 0.0
Max : 5.45245
Mean: 0.90425205
Std : 1.2472314

Percentiles:
 50.0% : 0.0
 90.0% : 3.002643
 95.0% : 3.452936
 99.0% : 4.6173673
 99.9% : 5.3107767
100.0% : 5.45245


**Layer 2's activation distribution is also well behaved.**

Layer 2 activation
Metric	Value
Range	0 → 5.45245
Mean	0.90425
Std	1.24723
90th percentile	3.00264
95th percentile	3.45294
99th percentile	4.61737
99.9th percentile	5.31078

The maximum is only slightly higher than Layer 1's, so the same full-range strategy is reasonable:

$$ S_{A2}=\frac{5.45245}{127} $$

Let's verify the actual quantization error rather than assuming it will be acceptable.

**Quantize Layer 2 activation to INT8**

In [25]:
# --------------------------------------------------
# Layer 2 activation → INT8
# --------------------------------------------------

A2_max = np.max(A2_fp32)

S_A2 = A2_max / 127.0

A2_int8 = np.round(
    A2_fp32 / S_A2
)

A2_int8 = np.clip(
    A2_int8,
    0,
    127
).astype(np.int8)

# Dequantize
A2_dequant = A2_int8.astype(np.float32) * S_A2

# Error
A2_error = A2_dequant - A2_fp32

print("Layer 2 activation quantization")
print("=" * 50)

print("Activation max :", A2_max)
print("Activation scale:", S_A2)

print("\nINT8 range:")
print("Min:", A2_int8.min())
print("Max:", A2_int8.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(A2_error)))
print("RMSE    :", np.sqrt(np.mean(A2_error ** 2)))
print("Max err :", np.max(np.abs(A2_error)))

print("\nSaturation count:")
print("127 values:", np.sum(A2_int8 == 127))

Layer 2 activation quantization
Activation max : 5.45245
Activation scale: 0.042932674

INT8 range:
Min: 0
Max: 127

Quantization error:
MAE     : 0.005321866
RMSE    : 0.008757118
Max err : 0.021458507

Saturation count:
127 values: 1


**Final Layer 3**

In [26]:
# --------------------------------------------------
# Layer 3 scales and bias
# --------------------------------------------------

S_Z3 = S_A2 * S_W3

b3_int32 = np.round(
    b3 / S_Z3
).astype(np.int32)

print("Layer 3")
print("=" * 50)

print("Activation scale S_A2 :", S_A2)
print("Weight scale S_W3    :", S_W3)
print("Accumulator scale    :", S_Z3)

print("\nFP32 bias:")
print("Min:", b3.min())
print("Max:", b3.max())

print("\nINT32 bias:")
print("Min:", b3_int32.min())
print("Max:", b3_int32.max())

# Bias reconstruction error
b3_deq = b3_int32.astype(np.float32) * S_Z3
b3_error = b3_deq - b3

print("\nBias quantization error:")
print("MAE     :", np.mean(np.abs(b3_error)))
print("RMSE    :", np.sqrt(np.mean(b3_error ** 2)))
print("Max err :", np.max(np.abs(b3_error)))

Layer 3
Activation scale S_A2 : 0.042932674
Weight scale S_W3    : 0.004470473
Accumulator scale    : 0.00019192937

FP32 bias:
Min: -0.025651688
Max: 0.02547786

INT32 bias:
Min: -134
Max: 133

Bias quantization error:
MAE     : 4.013651e-05
RMSE    : 4.5120112e-05
Max err : 6.684847e-05


**Integer final layer**

In [27]:
# --------------------------------------------------
# Integer Layer 3
# --------------------------------------------------

Z3_int = (
    A2_int8.astype(np.int32)
    @ W3_int8.astype(np.int32)
)

Z3_int_bias = Z3_int + b3_int32

print("\nInteger Layer 3")
print("=" * 50)

print("Shape :", Z3_int_bias.shape)
print("Dtype :", Z3_int_bias.dtype)
print("Min   :", Z3_int_bias.min())
print("Max   :", Z3_int_bias.max())
print("Mean  :", Z3_int_bias.mean())


Integer Layer 3
Shape : (412, 4)
Dtype : int32
Min   : -22389
Max   : 25830
Mean  : -1128.6984223300972


**Compare final logits**

In [29]:
# --------------------------------------------------
# Compare integer final logits with TensorFlow
# --------------------------------------------------

Z3_int_dequant = (
    Z3_int_bias.astype(np.float32) * S_Z3
)

# Original FP32 final layer
Z3_fp32 = A2_fp32 @ W3 + b3

error = Z3_int_dequant - Z3_fp32

print("\nFinal Layer 3 comparison")
print("=" * 50)

print("FP32 logits range:")
print("Min:", Z3_fp32.min())
print("Max:", Z3_fp32.max())

print("\nINT8/INT32 dequantized logits range:")
print("Min:", Z3_int_dequant.min())
print("Max:", Z3_int_dequant.max())

print("\nQuantization error:")
print("MAE     :", np.mean(np.abs(error)))
print("RMSE    :", np.sqrt(np.mean(error ** 2)))
print("Max err :", np.max(np.abs(error)))


Final Layer 3 comparison
FP32 logits range:
Min: -4.282553
Max: 4.97215

INT8/INT32 dequantized logits range:
Min: -4.2971067
Max: 4.9575357

Quantization error:
MAE     : 0.010463101
RMSE    : 0.013071313
Max err : 0.042538047


**compare the actual predicted class:**

In [30]:
tf_predictions = np.argmax(Z3_fp32, axis=1)

integer_predictions = np.argmax(
    Z3_int_bias,
    axis=1
)

print("TensorFlow accuracy:",
      np.mean(tf_predictions == y_test))

print("Integer accuracy:",
      np.mean(integer_predictions == y_test))

print("Prediction differences:",
      np.sum(tf_predictions != integer_predictions))

TensorFlow accuracy: 0.9830097087378641
Integer accuracy: 0.9830097087378641
Prediction differences: 0


In [ ]:
#Our complete quantization table

#We should save this information because it will become the specification for the HLS implementation.

#Quantity	Representation	Scale
#Input (X)	INT8	        0.031496063
#W1	        INT8            0.0025182366
#B1	        INT32	        0.00007931454
#A1	        INT8	        0.04329366
#W2	        INT8	        0.0033579213
#B2	        INT32	        0.0001453767
#A2	        INT8	        0.042932674
#W3	        INT8	        0.004470473
#B3	        INT32	        0.00019192937

In [33]:

#Load Stage 1 model                    ✅
#Load INT8 features                    ✅
#Recover input quantization            ✅
#Quantize W1/W2/W3                     ✅
#Validate W1 quantization              ✅
#Validate W2 quantization              ✅
#Validate W3 quantization              ✅

#Integer Layer 1                       ✅
#Integer Layer 1 + bias + ReLU         ✅

#Layer 1 → INT8                       ✅

#Integer Layer 2                       ✅
#Integer Layer 2 + bias + ReLU         ✅

#Layer 2 → INT8                       ✅

#Integer Layer 3                       ✅
#Integer final prediction              ✅

#FP32 accuracy                         98.30%
#Integer accuracy                      98.30%
#Prediction differences                0

**1. Create the integer_mlp() function**

In [36]:
def integer_mlp(X_int8):
    """
    Golden-reference integer implementation of the FOD MLP.

    Input:
        X_int8: shape (N, 64), dtype int8

    Returns:
        logits_int32: shape (N, 4), dtype int32
        predictions: shape (N,), class IDs
    """

    # ---------------------------------------------------------
    # Layer 1: 64 -> 32
    # ---------------------------------------------------------

    Z1 = (
        X_int8.astype(np.int32)
        @ W1_int8.astype(np.int32)
    )

    Z1 = Z1 + b1_int32

    # ReLU
    A1 = np.maximum(Z1, 0)

    # Requantize INT32 -> INT8
    A1_int8 = np.round(
        A1 * S_Z1 / S_A1
    ).astype(np.int8)


    # ---------------------------------------------------------
    # Layer 2: 32 -> 16
    # ---------------------------------------------------------

    Z2 = (
        A1_int8.astype(np.int32)
        @ W2_int8.astype(np.int32)
    )

    Z2 = Z2 + b2_int32

    # ReLU
    A2 = np.maximum(Z2, 0)

    # Requantize INT32 -> INT8
    A2_int8 = np.round(
        A2 * S_Z2 / S_A2
    ).astype(np.int8)


    # ---------------------------------------------------------
    # Layer 3: 16 -> 4
    # ---------------------------------------------------------

    Z3 = (
        A2_int8.astype(np.int32)
        @ W3_int8.astype(np.int32)
    )

    Z3 = Z3 + b3_int32


    # ---------------------------------------------------------
    # Classification
    # ---------------------------------------------------------

    predictions = np.argmax(Z3, axis=1)

    return Z3, predictions

**Test the function**

In [37]:
Z3_golden, pred_golden = integer_mlp(X_test_int8)

golden_accuracy = np.mean(pred_golden == y_test)

print("Golden-reference accuracy:", golden_accuracy)
print("Prediction differences:",
      np.sum(pred_golden != integer_predictions))

Golden-reference accuracy: 0.9830097087378641
Prediction differences: 0


**Check the exact output**

In [38]:
for i in range(10):
    print(
        f"Sample {i}: "
        f"True={y_test[i]}, "
        f"Pred={pred_golden[i]}, "
        f"Logits={Z3_golden[i]}"
    )

Sample 0: True=0, Pred=0, Logits=[  7933  -6744 -11895   1395]
Sample 1: True=0, Pred=0, Logits=[  7162  -5730 -11673   2894]
Sample 2: True=0, Pred=0, Logits=[  7315  -5952 -11489   2150]
Sample 3: True=0, Pred=0, Logits=[  8770  -7566 -12565   2197]
Sample 4: True=0, Pred=0, Logits=[  8949  -6722 -12688    862]
Sample 5: True=0, Pred=0, Logits=[  9688  -8520 -11358   -885]
Sample 6: True=0, Pred=0, Logits=[  9256  -7366 -11105   1028]
Sample 7: True=0, Pred=0, Logits=[  9554  -7814 -11339   1224]
Sample 8: True=0, Pred=0, Logits=[ 10373  -7902 -10662     57]
Sample 9: True=0, Pred=0, Logits=[ 10323  -8164 -11463    597]


**Add a consistency check**

In [42]:
assert Z3_golden.dtype == np.int32
assert pred_golden.dtype == np.int64 or pred_golden.dtype == np.int32

assert np.array_equal(
    pred_golden,
    integer_predictions
)

assert golden_accuracy == 0.9830097087378641

print("✓ Golden reference verified")
print("✓ INT32 logits")
print("✓ Predictions match previous implementation")
print("✓ Accuracy = 98.30%")

✓ Golden reference verified
✓ INT32 logits
✓ Predictions match previous implementation
✓ Accuracy = 98.30%


This function is going to be our software reference for the FPGA.

**The architecture is now:**

                  PYTHON
                    │
                    ▼
        ┌─────────────────────┐
        │ 64 INT8 features    │
        └──────────┬──────────┘
                   │
                   ▼
             64 × 32 MAC
                   │
                   ▼
             INT32 + bias
                   │
                 ReLU
                   │
                   ▼
               INT8
                   │
                   ▼
             32 × 16 MAC
                   │
                   ▼
             INT32 + bias
                   │
                 ReLU
                   │
                   ▼
               INT8
                   │
                   ▼
              16 × 4 MAC
                   │
                   ▼
                INT32
                   │
                   ▼
                ARGMAX
                   │
                   ▼
             CLASS 0–3

Later, Vitis HLS will implement exactly this computation.

We can then compare:

Python golden reference
          │
          │ same INT8 inputs
          ▼
      FPGA result

and verify:

Python prediction == FPGA prediction

That is much more useful than comparing the FPGA against the original floating-point TensorFlow model.

**Then generate mlp_weights.h**

In [46]:
from pathlib import Path

HEADER_PATH = r"D:\PROJECTS\64feature_NN\RGB_NN\weights\mlp_weights.h"

with open(HEADER_PATH, "w") as f:

    f.write("#ifndef MLP_WEIGHTS_H\n")
    f.write("#define MLP_WEIGHTS_H\n\n")

    f.write("#include <stdint.h>\n\n")

    # ---------------------------------------------------------
    # Dimensions
    # ---------------------------------------------------------

    f.write("#define INPUT_SIZE 64\n")
    f.write("#define HIDDEN1_SIZE 32\n")
    f.write("#define HIDDEN2_SIZE 16\n")
    f.write("#define OUTPUT_SIZE 4\n\n")

    # ---------------------------------------------------------
    # Quantization scales
    # ---------------------------------------------------------

    f.write(f"static const float S_X = {S_X:.12e}f;\n")
    f.write(f"static const float S_W1 = {S_W1:.12e}f;\n")
    f.write(f"static const float S_W2 = {S_W2:.12e}f;\n")
    f.write(f"static const float S_W3 = {S_W3:.12e}f;\n")

    f.write(f"static const float S_Z1 = {S_Z1:.12e}f;\n")
    f.write(f"static const float S_A1 = {S_A1:.12e}f;\n")
    f.write(f"static const float S_Z2 = {S_Z2:.12e}f;\n")
    f.write(f"static const float S_A2 = {S_A2:.12e}f;\n")
    f.write(f"static const float S_Z3 = {S_Z3:.12e}f;\n\n")

    # ---------------------------------------------------------
    # W1
    # ---------------------------------------------------------

    f.write(
        "static const int8_t W1[64][32] = {\n"
    )

    for row in W1_int8:
        f.write("    {")
        f.write(", ".join(str(int(x)) for x in row))
        f.write("},\n")

    f.write("};\n\n")

    # ---------------------------------------------------------
    # b1
    # ---------------------------------------------------------

    f.write(
        "static const int32_t b1[32] = {"
    )
    f.write(", ".join(str(int(x)) for x in b1_int32))
    f.write("};\n\n")

    # ---------------------------------------------------------
    # W2
    # ---------------------------------------------------------

    f.write(
        "static const int8_t W2[32][16] = {\n"
    )

    for row in W2_int8:
        f.write("    {")
        f.write(", ".join(str(int(x)) for x in row))
        f.write("},\n")

    f.write("};\n\n")

    # ---------------------------------------------------------
    # b2
    # ---------------------------------------------------------

    f.write(
        "static const int32_t b2[16] = {"
    )
    f.write(", ".join(str(int(x)) for x in b2_int32))
    f.write("};\n\n")

    # ---------------------------------------------------------
    # W3
    # ---------------------------------------------------------

    f.write(
        "static const int8_t W3[16][4] = {\n"
    )

    for row in W3_int8:
        f.write("    {")
        f.write(", ".join(str(int(x)) for x in row))
        f.write("},\n")

    f.write("};\n\n")

    # ---------------------------------------------------------
    # b3
    # ---------------------------------------------------------

    f.write(
        "static const int32_t b3[4] = {"
    )
    f.write(", ".join(str(int(x)) for x in b3_int32))
    f.write("};\n\n")

    f.write("#endif\n")


print("Created:", HEADER_PATH)

Created: D:\PROJECTS\64feature_NN\RGB_NN\weights\mlp_weights.h


In [47]:
header_path = PROJECT_ROOT / "weights" / "mlp_weights.h"

print("Header exists:", header_path.exists())
print("Header size:", header_path.stat().st_size, "bytes")

Header exists: True
Header size: 13640 bytes


# **The next step is Vitis HLS C Simulation**

In [48]:
Stage 2 checkpoint

You now have:

✅ Stage 1 lightweight RGB FOD classifier
✅ 64 engineered features
✅ FP32 MLP: 98.30% test accuracy
✅ INT8 input quantization verified
✅ INT8 weights + INT32 biases implemented
✅ Full integer Python MLP: 98.30%
✅ Integer predictions exactly match TensorFlow: 0 differences
✅ FPGA weight header generated:
weights\mlp_weights.h

SyntaxError: invalid character '✅' (U+2705) (1469565386.py, line 5)